[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week07_multiple_testing.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 7: Multiple Testing & Experiment Infrastructure

**Course**: Advanced A/B Testing | **Context**: QuickCart (online electronics retailer)

**Your role**: Data scientist on the experimentation platform team. QuickCart has matured from running one experiment at a time to launching **10+ concurrent experiments** across search, checkout, recommendations, and pricing. This week you confront two scaling challenges: (1) the multiple comparisons problem when each experiment measures dozens of metrics, and (2) building the infrastructure to split users across concurrent experiments deterministically.

---

## What You'll Learn

1. The multiple comparisons problem (why testing 20 metrics inflates false positives)
2. FWER vs FDR: which to control and when
3. Bonferroni correction (simple but conservative)
4. Holm's step-down procedure (uniformly more powerful than Bonferroni)
5. Benjamini-Hochberg FDR control (for exploratory analysis)
6. Simulations showing each method's behavior
7. Experiment infrastructure: why hash-based splitting matters
8. Implementing a slot-based experiment allocator
9. Hash-based user assignment (deterministic, reproducible)
10. Handling multiple concurrent experiments

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import hashlib

from itertools import combinations

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. The Multiple Comparisons Problem

### Why Testing 20 Metrics Inflates False Positives

QuickCart's experimentation team tracks **20 metrics** for every experiment: revenue, conversion rate, AOV, page load time, bounce rate, search CTR, add-to-cart rate, and so on. Even when the treatment has **no real effect**, testing 20 metrics at $\alpha = 0.05$ each gives:

$$P(\text{at least one false positive}) = 1 - (1 - 0.05)^{20} = 1 - 0.95^{20} \approx 0.64$$

That means **64% of null experiments will produce at least one "significant" metric** by chance alone. This is the **familywise error rate (FWER)** inflation.

### The Intuition

Each test is a 5% lottery ticket for a false positive. Buy 20 tickets, and you're very likely to "win" at least once. The more metrics you test, the more likely you are to find spurious results.

In [ ]:
# Demonstrate FWER inflation
n_metrics_range = np.arange(1, 51)
alpha = 0.05

# Probability of at least one false positive across m independent tests
fwer_theoretical = 1 - (1 - alpha) ** n_metrics_range

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_metrics_range, fwer_theoretical, 'b-', linewidth=2.5)
ax.axhline(0.05, color='red', linestyle='--', linewidth=2, label='Target FWER = 0.05')
ax.axhline(0.64, color='gray', linestyle=':', alpha=0.7)
ax.axvline(20, color='gray', linestyle=':', alpha=0.7)
ax.scatter([20], [fwer_theoretical[19]], color='red', s=100, zorder=5)
ax.annotate(f'20 metrics: FWER = {fwer_theoretical[19]:.2f}',
            xy=(20, fwer_theoretical[19]), xytext=(25, 0.55),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Number of Metrics Tested')
ax.set_ylabel('P(at least one false positive)')
ax.set_title('Familywise Error Rate vs Number of Tests')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim([0, 1.0])
plt.tight_layout()
plt.show()

print(f"With 20 metrics at alpha=0.05 each:")
print(f"  P(at least one false positive) = {fwer_theoretical[19]:.3f}")
print(f"  Expected number of false positives = {20 * 0.05:.1f}")

In [ ]:
# Simulation: run 10,000 null experiments, each testing 20 metrics
n_simulations = 10000
n_metrics = 20
n_users = 1000  # per group

any_significant = 0
total_false_positives = 0

for _ in range(n_simulations):
    # Generate 20 metrics for control and treatment (all from same distribution = null)
    # Metrics are correlated within user (realistic)
    correlation = 0.3
    cov_matrix = np.full((n_metrics, n_metrics), correlation)
    np.fill_diagonal(cov_matrix, 1.0)
    
    control = np.random.multivariate_normal(np.zeros(n_metrics), cov_matrix, n_users)
    treatment = np.random.multivariate_normal(np.zeros(n_metrics), cov_matrix, n_users)
    
    # Test each metric
    pvalues = []
    for m in range(n_metrics):
        _, p = stats.ttest_ind(treatment[:, m], control[:, m])
        pvalues.append(p)
    
    pvalues = np.array(pvalues)
    n_sig = np.sum(pvalues < 0.05)
    total_false_positives += n_sig
    if n_sig > 0:
        any_significant += 1

print(f"Simulation results ({n_simulations:,} null experiments, {n_metrics} correlated metrics each):")
print(f"  Experiments with at least one false positive: {any_significant/n_simulations:.3f}")
print(f"  Average false positives per experiment:       {total_false_positives/n_simulations:.2f}")
print(f"  Theoretical FWER (independent):              {1-(1-0.05)**20:.3f}")
print(f"\nNote: With correlated metrics (rho=0.3), FWER is lower than the")
print(f"independent case but still far above 0.05.")

---

## 2. FWER vs FDR: Which to Control and When

Two frameworks for handling multiple comparisons:

### Familywise Error Rate (FWER)

$$\text{FWER} = P(\text{at least one false positive among all tests})$$

**When to use**: High-stakes decisions where *any* false positive is costly.
- Launching a feature based on a guardrail metric
- Regulatory submissions
- Primary metrics that drive ship/no-ship decisions

**Methods**: Bonferroni, Holm's step-down

### False Discovery Rate (FDR)

$$\text{FDR} = E\left[\frac{\text{false positives}}{\text{total rejections}}\right]$$

**When to use**: Exploratory analysis where some false discoveries are acceptable if you find most of the true effects.
- Screening 50 secondary metrics for insights
- Identifying which user segments show an effect
- Hypothesis generation for follow-up experiments

**Methods**: Benjamini-Hochberg

### QuickCart's Framework

| Metric Type | Error Control | Method | Rationale |
|-------------|---------------|--------|-----------|
| Primary (revenue, conversion) | FWER at 0.05 | Holm's | Ship/no-ship decision |
| Guardrail (latency, crash rate) | FWER at 0.05 | Bonferroni | Any degradation is bad |
| Secondary (engagement, CTR) | FDR at 0.10 | BH | Exploratory insights |

---

## 3. Bonferroni Correction

The simplest FWER correction: divide the significance level by the number of tests.

$$\alpha_{\text{adjusted}} = \frac{\alpha}{m}$$

Reject $H_i$ if $p_i < \alpha/m$.

**Why it works** (Boole's inequality): For any set of events $A_1, \ldots, A_m$:

$$P\left(\bigcup_{i=1}^m A_i\right) \leq \sum_{i=1}^m P(A_i) = m \cdot \frac{\alpha}{m} = \alpha$$

**Pros**: Simple, valid for any dependence structure between tests.

**Cons**: Very conservative when $m$ is large. With 20 metrics, the per-test threshold drops to $0.05/20 = 0.0025$.

In [ ]:
def bonferroni_correction(pvalues, alpha=0.05):
    """
    Apply Bonferroni correction to a set of p-values.
    
    Parameters
    ----------
    pvalues : array-like
        Raw p-values from multiple tests
    alpha : float
        Target FWER
    
    Returns
    -------
    dict with adjusted threshold, rejections, and adjusted p-values
    """
    pvalues = np.asarray(pvalues)
    m = len(pvalues)
    threshold = alpha / m
    
    # Adjusted p-values: min(p_i * m, 1)
    adjusted_pvalues = np.minimum(pvalues * m, 1.0)
    rejections = pvalues < threshold
    
    return {
        'threshold': threshold,
        'rejections': rejections,
        'adjusted_pvalues': adjusted_pvalues,
        'n_rejected': rejections.sum()
    }


# Example: QuickCart tests 20 metrics, 3 have real effects
np.random.seed(42)
n_metrics = 20
n_true_effects = 3

# Simulate p-values: first 3 are from real effects, rest are null
pvalues_real = stats.beta.rvs(1, 50, size=n_true_effects)  # small p-values
pvalues_null = np.random.uniform(0, 1, size=n_metrics - n_true_effects)
pvalues_all = np.concatenate([pvalues_real, pvalues_null])

print(f"Raw p-values (first 3 are TRUE effects):")
for i, p in enumerate(pvalues_all):
    marker = " <-- TRUE EFFECT" if i < n_true_effects else ""
    sig = " *" if p < 0.05 else ""
    print(f"  Metric {i+1:2d}: p = {p:.4f}{sig}{marker}")

# Apply Bonferroni
result_bonf = bonferroni_correction(pvalues_all, alpha=0.05)
print(f"\nBonferroni correction (threshold = {result_bonf['threshold']:.4f}):")
print(f"  Rejected: {result_bonf['n_rejected']} metrics")
print(f"  Without correction: {np.sum(pvalues_all < 0.05)} metrics would be 'significant'")

---

## 4. Holm's Step-Down Procedure

Holm's method is **uniformly more powerful** than Bonferroni while still controlling FWER. It rejects at least as many hypotheses as Bonferroni in every case.

### Algorithm

1. Sort p-values: $p_{(1)} \leq p_{(2)} \leq \ldots \leq p_{(m)}$
2. For $k = 1, 2, \ldots, m$:
   - If $p_{(k)} < \frac{\alpha}{m - k + 1}$, reject $H_{(k)}$ and continue
   - Otherwise, stop. Do not reject $H_{(k)}$ or any remaining hypotheses.

### Why It's More Powerful

Bonferroni uses a fixed threshold $\alpha/m$ for all tests. Holm's uses $\alpha/m$ only for the smallest p-value, then $\alpha/(m-1)$ for the next, then $\alpha/(m-2)$, etc. As you reject hypotheses, the threshold becomes more lenient for the remaining ones.

:::{admonition} Why Holm's controls FWER -- proof sketch (click to expand)
:class: dropdown

Let $m_0$ be the (unknown) number of true null hypotheses. We want to show that the probability of rejecting any true null is at most $\alpha$.

Consider the true null hypothesis with the smallest p-value. Call it $H_{(j)}$ where $j$ is its position in the sorted order. For Holm's to falsely reject it, all of $p_{(1)}, \ldots, p_{(j)}$ must pass their respective thresholds.

In particular, $p_{(j)}$ must satisfy $p_{(j)} < \alpha/(m - j + 1)$. Since there are $m_0$ true nulls, $m - j + 1 \geq m_0$, so the threshold is at most $\alpha/m_0$.

By the union bound over $m_0$ true nulls:

$$\text{FWER} \leq m_0 \cdot \frac{\alpha}{m_0} = \alpha$$

This holds regardless of the dependence structure between tests.
:::

In [ ]:
def holm_correction(pvalues, alpha=0.05):
    """
    Apply Holm's step-down procedure.
    
    Parameters
    ----------
    pvalues : array-like
        Raw p-values from multiple tests
    alpha : float
        Target FWER
    
    Returns
    -------
    dict with rejections, adjusted p-values, and step thresholds
    """
    pvalues = np.asarray(pvalues)
    m = len(pvalues)
    
    # Sort p-values and track original indices
    sorted_indices = np.argsort(pvalues)
    sorted_pvalues = pvalues[sorted_indices]
    
    # Compute step-down thresholds
    thresholds = alpha / (m - np.arange(m))
    
    # Find the first index where p >= threshold (stop rejecting)
    rejections_sorted = np.zeros(m, dtype=bool)
    for k in range(m):
        if sorted_pvalues[k] < thresholds[k]:
            rejections_sorted[k] = True
        else:
            break  # Stop: don't reject this or any remaining
    
    # Map back to original order
    rejections = np.zeros(m, dtype=bool)
    rejections[sorted_indices] = rejections_sorted
    
    # Compute adjusted p-values
    adjusted_sorted = np.zeros(m)
    for k in range(m):
        adjusted_sorted[k] = sorted_pvalues[k] * (m - k)
    # Enforce monotonicity: adjusted p-values must be non-decreasing
    for k in range(1, m):
        adjusted_sorted[k] = max(adjusted_sorted[k], adjusted_sorted[k-1])
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)
    
    adjusted_pvalues = np.zeros(m)
    adjusted_pvalues[sorted_indices] = adjusted_sorted
    
    return {
        'rejections': rejections,
        'adjusted_pvalues': adjusted_pvalues,
        'n_rejected': rejections.sum(),
        'thresholds': thresholds
    }


# Compare Bonferroni vs Holm on the same p-values
result_holm = holm_correction(pvalues_all, alpha=0.05)

print(f"Comparison on {n_metrics} metrics ({n_true_effects} true effects):")
print(f"\n{'Metric':<10} {'p-value':<10} {'Bonferroni':<14} {'Holm':<14} {'Truth':<10}")
print("-" * 58)

sorted_idx = np.argsort(pvalues_all)
for rank, i in enumerate(sorted_idx[:10]):  # Show top 10
    bonf_decision = 'Reject' if result_bonf['rejections'][i] else 'Retain'
    holm_decision = 'Reject' if result_holm['rejections'][i] else 'Retain'
    truth = 'Effect' if i < n_true_effects else 'Null'
    print(f"  {i+1:<8} {pvalues_all[i]:<10.4f} {bonf_decision:<14} {holm_decision:<14} {truth:<10}")

print(f"\nTotal rejections: Bonferroni = {result_bonf['n_rejected']}, Holm = {result_holm['n_rejected']}")
print(f"Holm always rejects >= Bonferroni (uniformly more powerful).")

---

## 5. Benjamini-Hochberg FDR Control

When you're running an exploratory analysis (screening many secondary metrics), controlling FWER is too conservative. You'd rather allow some false discoveries as long as the *proportion* of false discoveries among all discoveries is controlled.

### Algorithm

1. Sort p-values: $p_{(1)} \leq p_{(2)} \leq \ldots \leq p_{(m)}$
2. Find the largest $k$ such that $p_{(k)} \leq \frac{k}{m} \cdot q$ (where $q$ is the target FDR)
3. Reject all $H_{(1)}, \ldots, H_{(k)}$

### Geometric Interpretation

Plot the sorted p-values against the BH line $y = (k/m) \cdot q$. Reject everything to the left of the last p-value that falls below the line.

:::{admonition} Why Benjamini-Hochberg controls FDR (click to expand)
:class: dropdown

Let $m_0$ be the number of true nulls and $m_1 = m - m_0$ be the number of true alternatives. The BH procedure rejects all hypotheses with $p_{(k)} \leq (k/m) \cdot q$.

Under independence of the test statistics:

$$\text{FDR} = \frac{m_0}{m} \cdot q \leq q$$

The key insight: p-values for true nulls are Uniform[0,1], so they cross the BH line $y = (k/m) \cdot q$ at a predictable rate. Among the $k$ rejected hypotheses, the expected number of false discoveries is approximately $m_0 \cdot (k/m) \cdot q / 1 = k \cdot (m_0/m) \cdot q$. Dividing by $k$ total rejections gives FDR $\approx (m_0/m) \cdot q \leq q$.

The formal proof by Benjamini and Hochberg (1995) uses a careful conditioning argument. The result also holds under certain forms of positive dependence (the PRDS condition), which is satisfied when test statistics are positively correlated -- common in A/B testing.
:::

In [ ]:
def benjamini_hochberg(pvalues, q=0.10):
    """
    Apply Benjamini-Hochberg FDR control.
    
    Parameters
    ----------
    pvalues : array-like
        Raw p-values from multiple tests
    q : float
        Target FDR level
    
    Returns
    -------
    dict with rejections, adjusted p-values, and critical values
    """
    pvalues = np.asarray(pvalues)
    m = len(pvalues)
    
    # Sort p-values
    sorted_indices = np.argsort(pvalues)
    sorted_pvalues = pvalues[sorted_indices]
    
    # BH critical values: (k/m) * q
    ranks = np.arange(1, m + 1)
    critical_values = (ranks / m) * q
    
    # Find the largest k where p_(k) <= critical_value_k
    below_line = sorted_pvalues <= critical_values
    if below_line.any():
        max_k = np.max(np.where(below_line)[0])  # 0-indexed
        rejections_sorted = np.zeros(m, dtype=bool)
        rejections_sorted[:max_k + 1] = True
    else:
        rejections_sorted = np.zeros(m, dtype=bool)
    
    # Map back to original order
    rejections = np.zeros(m, dtype=bool)
    rejections[sorted_indices] = rejections_sorted
    
    # Adjusted p-values (BH adjusted)
    adjusted_sorted = np.zeros(m)
    adjusted_sorted[-1] = sorted_pvalues[-1]
    for k in range(m - 2, -1, -1):
        adjusted_sorted[k] = min(adjusted_sorted[k + 1], sorted_pvalues[k] * m / (k + 1))
    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)
    
    adjusted_pvalues = np.zeros(m)
    adjusted_pvalues[sorted_indices] = adjusted_sorted
    
    return {
        'rejections': rejections,
        'adjusted_pvalues': adjusted_pvalues,
        'n_rejected': rejections.sum(),
        'critical_values': critical_values,
        'sorted_pvalues': sorted_pvalues,
        'sorted_indices': sorted_indices
    }


# Apply BH to our example
result_bh = benjamini_hochberg(pvalues_all, q=0.10)

print(f"Benjamini-Hochberg at FDR = 0.10:")
print(f"  Rejected: {result_bh['n_rejected']} metrics")
print(f"\nAll three methods compared:")
print(f"  Uncorrected (alpha=0.05): {np.sum(pvalues_all < 0.05)} rejections")
print(f"  Bonferroni (FWER=0.05):   {result_bonf['n_rejected']} rejections")
print(f"  Holm (FWER=0.05):         {result_holm['n_rejected']} rejections")
print(f"  BH (FDR=0.10):            {result_bh['n_rejected']} rejections")

In [ ]:
# Visualize BH procedure: sorted p-values vs BH line
fig, ax = plt.subplots(figsize=(10, 6))

m = len(pvalues_all)
ranks = np.arange(1, m + 1)
sorted_pvals = result_bh['sorted_pvalues']
bh_line = (ranks / m) * 0.10

# Color points by true status
sorted_idx = result_bh['sorted_indices']
is_true_effect = sorted_idx < n_true_effects

ax.scatter(ranks[is_true_effect], sorted_pvals[is_true_effect], 
           color='green', s=80, zorder=5, label='True effects', marker='o')
ax.scatter(ranks[~is_true_effect], sorted_pvals[~is_true_effect],
           color='gray', s=80, zorder=5, label='Null metrics', marker='x')

ax.plot(ranks, bh_line, 'b-', linewidth=2, label='BH line (FDR=0.10)')
ax.axhline(0.05 / m, color='red', linestyle='--', linewidth=2, 
           label=f'Bonferroni threshold ({0.05/m:.4f})')

# Shade rejected region
if result_bh['n_rejected'] > 0:
    ax.axvspan(0.5, result_bh['n_rejected'] + 0.5, alpha=0.1, color='blue',
               label=f'BH rejections (k={result_bh["n_rejected"]})')

ax.set_xlabel('Rank (sorted p-values)')
ax.set_ylabel('p-value')
ax.set_title('Benjamini-Hochberg Procedure: Sorted P-values vs Critical Line')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim([0.5, m + 0.5])
plt.tight_layout()
plt.show()

---

## 6. Simulation: Comparing All Methods

Let's run a large simulation to compare FWER, power, and FDR for each correction method across different scenarios.

In [ ]:
def run_multiple_testing_simulation(n_metrics=20, n_true_effects=5, 
                                     effect_size=0.3, n_users=500,
                                     n_simulations=5000):
    """
    Simulate multiple testing scenarios and compare correction methods.
    
    Parameters
    ----------
    n_metrics : number of metrics tested simultaneously
    n_true_effects : how many metrics have real effects
    effect_size : Cohen's d for the real effects
    n_users : sample size per group
    n_simulations : number of simulation runs
    """
    results = {
        'uncorrected': {'fp': 0, 'tp': 0, 'fn': 0, 'tn': 0, 'any_fp': 0},
        'bonferroni': {'fp': 0, 'tp': 0, 'fn': 0, 'tn': 0, 'any_fp': 0},
        'holm': {'fp': 0, 'tp': 0, 'fn': 0, 'tn': 0, 'any_fp': 0},
        'bh': {'fp': 0, 'tp': 0, 'fn': 0, 'tn': 0, 'any_fp': 0},
    }
    
    for _ in range(n_simulations):
        # Generate p-values
        pvalues = np.zeros(n_metrics)
        truth = np.zeros(n_metrics, dtype=bool)  # True = has real effect
        
        for m_idx in range(n_metrics):
            control = np.random.normal(0, 1, n_users)
            if m_idx < n_true_effects:
                treatment = np.random.normal(effect_size, 1, n_users)
                truth[m_idx] = True
            else:
                treatment = np.random.normal(0, 1, n_users)
            _, pvalues[m_idx] = stats.ttest_ind(treatment, control)
        
        # Apply each method
        methods_decisions = {
            'uncorrected': pvalues < 0.05,
            'bonferroni': bonferroni_correction(pvalues)['rejections'],
            'holm': holm_correction(pvalues)['rejections'],
            'bh': benjamini_hochberg(pvalues, q=0.10)['rejections'],
        }
        
        for method, decisions in methods_decisions.items():
            fp = np.sum(decisions & ~truth)
            tp = np.sum(decisions & truth)
            fn = np.sum(~decisions & truth)
            tn = np.sum(~decisions & ~truth)
            
            results[method]['fp'] += fp
            results[method]['tp'] += tp
            results[method]['fn'] += fn
            results[method]['tn'] += tn
            if fp > 0:
                results[method]['any_fp'] += 1
    
    # Compute summary statistics
    summary = {}
    for method, counts in results.items():
        total_rejections = counts['fp'] + counts['tp']
        fdr = counts['fp'] / total_rejections if total_rejections > 0 else 0
        power = counts['tp'] / (counts['tp'] + counts['fn']) if (counts['tp'] + counts['fn']) > 0 else 0
        fwer = counts['any_fp'] / n_simulations
        
        summary[method] = {
            'FWER': fwer,
            'FDR': fdr,
            'Power': power,
            'Avg_Rejections': total_rejections / n_simulations
        }
    
    return summary


# Run simulation
print("Running simulation: 20 metrics, 5 true effects, Cohen's d=0.3, n=500/group...")
summary = run_multiple_testing_simulation(
    n_metrics=20, n_true_effects=5, effect_size=0.3, 
    n_users=500, n_simulations=5000
)

print(f"\n{'Method':<15} {'FWER':<8} {'FDR':<8} {'Power':<8} {'Avg Rejections':<15}")
print("-" * 54)
for method, stats_dict in summary.items():
    print(f"{method:<15} {stats_dict['FWER']:<8.3f} {stats_dict['FDR']:<8.3f} "
          f"{stats_dict['Power']:<8.3f} {stats_dict['Avg_Rejections']:<15.2f}")

In [ ]:
# Visualize: how does each method perform as we vary effect size?
effect_sizes = [0.1, 0.2, 0.3, 0.4, 0.5]
power_by_method = {m: [] for m in ['uncorrected', 'bonferroni', 'holm', 'bh']}
fwer_by_method = {m: [] for m in ['uncorrected', 'bonferroni', 'holm', 'bh']}

print("Running simulations across effect sizes...")
for es in effect_sizes:
    s = run_multiple_testing_simulation(
        n_metrics=20, n_true_effects=5, effect_size=es,
        n_users=500, n_simulations=3000
    )
    for method in power_by_method:
        power_by_method[method].append(s[method]['Power'])
        fwer_by_method[method].append(s[method]['FWER'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'uncorrected': 'red', 'bonferroni': 'blue', 'holm': 'green', 'bh': 'orange'}
labels = {'uncorrected': 'Uncorrected', 'bonferroni': 'Bonferroni', 
          'holm': 'Holm', 'bh': 'BH (FDR=0.10)'}

for method in power_by_method:
    axes[0].plot(effect_sizes, power_by_method[method], 'o-', 
                 color=colors[method], linewidth=2, label=labels[method])
axes[0].axhline(0.80, color='black', linestyle=':', linewidth=1.5, label='80% power target')
axes[0].set_xlabel('Effect Size (Cohen\'s d)')
axes[0].set_ylabel('Power (per true effect)')
axes[0].set_title('Statistical Power vs Effect Size')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_ylim([0, 1.0])

for method in fwer_by_method:
    axes[1].plot(effect_sizes, fwer_by_method[method], 'o-',
                 color=colors[method], linewidth=2, label=labels[method])
axes[1].axhline(0.05, color='black', linestyle=':', linewidth=1.5, label='Target FWER = 0.05')
axes[1].set_xlabel('Effect Size (Cohen\'s d)')
axes[1].set_ylabel('FWER')
axes[1].set_title('Familywise Error Rate vs Effect Size')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_ylim([0, 0.8])

plt.tight_layout()
plt.show()

print("Key takeaways:")
print("- Bonferroni and Holm control FWER at 0.05 regardless of effect size")
print("- Holm always has >= power compared to Bonferroni")
print("- BH has much higher power but allows higher FWER")
print("- The power gap between methods shrinks as effect size grows")

In [ ]:
# QuickCart practical example: analyzing a real experiment result
np.random.seed(123)

# QuickCart ran a recommendation algorithm change
# They measure 20 metrics; the treatment truly improves 4 of them
metric_names = [
    'Revenue', 'Conversion Rate', 'AOV', 'Sessions per User',
    'Search CTR', 'Add-to-Cart Rate', 'Bounce Rate', 'Page Load Time',
    'Pages per Session', 'Time on Site', 'Return Rate', 'Cart Abandonment',
    'Wishlist Adds', 'Review Submissions', 'Support Tickets', 'App Crashes',
    'Email Unsubscribes', 'Push Notification CTR', 'Share Actions', 'NPS Score'
]

# Simulate: first 4 metrics have real effects of varying sizes
true_effects = [0.15, 0.20, 0.12, 0.10]  # Cohen's d values
n_per_group = 5000

pvalues_quickcart = []
observed_effects = []

for i in range(20):
    control = np.random.normal(0, 1, n_per_group)
    if i < 4:
        treatment = np.random.normal(true_effects[i], 1, n_per_group)
    else:
        treatment = np.random.normal(0, 1, n_per_group)
    _, p = stats.ttest_ind(treatment, control)
    pvalues_quickcart.append(p)
    observed_effects.append(treatment.mean() - control.mean())

pvalues_quickcart = np.array(pvalues_quickcart)

# Apply all corrections
bonf_result = bonferroni_correction(pvalues_quickcart)
holm_result = holm_correction(pvalues_quickcart)
bh_result = benjamini_hochberg(pvalues_quickcart, q=0.10)

print("QuickCart Recommendation Experiment Results")
print("=" * 85)
print(f"\n{'Metric':<22} {'Effect':<8} {'p-value':<10} {'Raw':<6} {'Bonf':<6} {'Holm':<6} {'BH':<6} {'Truth':<8}")
print("-" * 85)

for i in range(20):
    raw_sig = '*' if pvalues_quickcart[i] < 0.05 else ' '
    bonf_sig = '*' if bonf_result['rejections'][i] else ' '
    holm_sig = '*' if holm_result['rejections'][i] else ' '
    bh_sig = '*' if bh_result['rejections'][i] else ' '
    truth = 'REAL' if i < 4 else 'null'
    
    print(f"{metric_names[i]:<22} {observed_effects[i]:>+.4f} {pvalues_quickcart[i]:<10.4f} "
          f"{raw_sig:<6} {bonf_sig:<6} {holm_sig:<6} {bh_sig:<6} {truth:<8}")

print(f"\n{'Method':<20} {'Rejections':<12} {'True Positives':<16} {'False Positives':<16}")
print("-" * 64)
for name, result in [('Uncorrected', pvalues_quickcart < 0.05), 
                      ('Bonferroni', bonf_result['rejections']),
                      ('Holm', holm_result['rejections']),
                      ('BH (FDR=0.10)', bh_result['rejections'])]:
    tp = np.sum(result[:4])
    fp = np.sum(result[4:])
    print(f"{name:<20} {np.sum(result):<12} {tp:<16} {fp:<16}")

---

## 7. Experiment Infrastructure: Why Hash-Based Splitting Matters

### The Scaling Problem

QuickCart now runs 10+ concurrent experiments. Naive approaches to user assignment fail:

- **Random assignment at request time**: Not deterministic. The same user might see treatment on one visit and control on the next.
- **Database lookup**: Requires storing assignments for millions of users x experiments. Slow, expensive, single point of failure.
- **Modulo user_id**: Only works for one experiment. How do you assign the same user to independent experiments?

### Requirements for Production Experiment Infrastructure

1. **Deterministic**: Same user always gets the same assignment (no matter when or how many times they visit)
2. **Uniform**: Each group gets approximately equal traffic
3. **Independent across experiments**: User's assignment to Experiment A doesn't correlate with their assignment to Experiment B
4. **Scalable**: Works for millions of users, no database required at assignment time
5. **Non-overlapping**: Experiments can be isolated to subsets of traffic (slots)

### The Solution: Hash-Based Splitting

Use cryptographic hashing (MD5) to map `(user_id, experiment_id)` to a pseudo-random number. This gives you:
- Determinism (same input = same hash)
- Uniformity (hashes are uniformly distributed)
- Independence (different experiment_ids produce uncorrelated hashes)

---

## 8. Implementing a Slot-Based Experiment Allocator

The **slot system** divides traffic into 100 slots (0-99). Each experiment is allocated a contiguous range of slots. A user's slot is determined by hashing their user_id.

This ensures:
- Experiments don't overlap (isolated traffic)
- Slot allocation is deterministic
- We can run experiments on any fraction of traffic (e.g., 10 slots = 10% of users)

In [ ]:
class ABSplitter:
    """
    Hash-based experiment traffic splitter with slot allocation.
    
    Uses MD5 double-hashing:
    - First hash: user_id -> slot (0-99) for traffic isolation
    - Second hash: user_id + experiment_id -> group (A/B) for assignment
    
    Parameters
    ----------
    count_slots : int
        Total number of traffic slots (default 100)
    """
    
    def __init__(self, count_slots=100):
        self.count_slots = count_slots
        self.experiments = {}  # experiment_id -> (start_slot, end_slot)
        self.slot_map = [None] * count_slots  # slot -> experiment_id or None
    
    def split_experiments(self, experiments):
        """
        Assign contiguous slot ranges to each experiment.
        
        Parameters
        ----------
        experiments : list of tuples
            Each tuple is (experiment_id, num_slots_needed)
            e.g., [('search_v2', 20), ('checkout_flow', 30)]
        
        Returns
        -------
        dict : experiment_id -> (start_slot, end_slot)
        
        Raises
        ------
        ValueError : if total slots requested exceeds available slots
        """
        total_needed = sum(n for _, n in experiments)
        if total_needed > self.count_slots:
            raise ValueError(
                f"Total slots requested ({total_needed}) exceeds available ({self.count_slots})"
            )
        
        # Reset
        self.experiments = {}
        self.slot_map = [None] * self.count_slots
        
        # Assign contiguous slot ranges
        current_slot = 0
        for exp_id, num_slots in experiments:
            start = current_slot
            end = current_slot + num_slots - 1
            self.experiments[exp_id] = (start, end)
            
            for s in range(start, end + 1):
                self.slot_map[s] = exp_id
            
            current_slot = end + 1
        
        return self.experiments
    
    def _hash_to_slot(self, user_id):
        """
        First hash: map user_id to a traffic slot (0 to count_slots-1).
        Uses MD5 for uniform distribution.
        """
        hash_input = str(user_id).encode('utf-8')
        hash_hex = hashlib.md5(hash_input).hexdigest()
        # Use first 8 hex chars (32 bits) for slot assignment
        hash_int = int(hash_hex[:8], 16)
        return hash_int % self.count_slots
    
    def _hash_to_group(self, user_id, experiment_id):
        """
        Second hash: map (user_id, experiment_id) to group A or B.
        Independent of the slot hash because it uses a different input.
        """
        hash_input = f"{user_id}_{experiment_id}".encode('utf-8')
        hash_hex = hashlib.md5(hash_input).hexdigest()
        # Use last 8 hex chars (different bits from slot hash)
        hash_int = int(hash_hex[-8:], 16)
        return 'B' if hash_int % 2 == 0 else 'A'
    
    def process_user(self, user_id):
        """
        Determine which experiments a user is in and their group assignment.
        
        Double-hash approach:
        1. Hash user_id to determine their slot
        2. Check which experiment (if any) owns that slot
        3. Hash user_id + experiment_id to determine A/B group
        
        Parameters
        ----------
        user_id : str or int
            Unique user identifier
        
        Returns
        -------
        dict : {experiment_id: 'A' or 'B'} for experiments this user is in
              Empty dict if user's slot is unallocated
        """
        slot = self._hash_to_slot(user_id)
        experiment_id = self.slot_map[slot]
        
        if experiment_id is None:
            return {}  # User is not in any experiment
        
        group = self._hash_to_group(user_id, experiment_id)
        return {experiment_id: group}
    
    def get_allocation_summary(self):
        """Print a summary of slot allocations."""
        print(f"Slot Allocation ({self.count_slots} total slots):")
        print("-" * 50)
        for exp_id, (start, end) in self.experiments.items():
            n_slots = end - start + 1
            pct = n_slots / self.count_slots * 100
            print(f"  {exp_id:<20} slots {start:2d}-{end:2d} ({n_slots} slots, {pct:.0f}% traffic)")
        
        allocated = sum(1 for s in self.slot_map if s is not None)
        print(f"\n  Allocated: {allocated}/{self.count_slots} slots")
        print(f"  Available: {self.count_slots - allocated} slots")

In [ ]:
# Demo: QuickCart runs 4 concurrent experiments
splitter = ABSplitter(count_slots=100)

# Allocate experiments to traffic slots
experiments = [
    ('search_ranking_v2', 20),    # 20% of traffic
    ('checkout_redesign', 30),    # 30% of traffic
    ('rec_algorithm', 20),        # 20% of traffic
    ('pricing_test', 10),         # 10% of traffic
]

allocations = splitter.split_experiments(experiments)
splitter.get_allocation_summary()

# Process some sample users
print(f"\nSample user assignments:")
print(f"{'User ID':<15} {'Slot':<6} {'Assignment':<30}")
print("-" * 51)
for uid in ['user_001', 'user_042', 'user_100', 'user_777', 'user_999']:
    slot = splitter._hash_to_slot(uid)
    assignment = splitter.process_user(uid)
    assign_str = str(assignment) if assignment else '(no experiment)'
    print(f"{uid:<15} {slot:<6} {assign_str:<30}")

---

## 9. Hash-Based User Assignment: Properties and Validation

Let's verify that our hash-based splitting satisfies the key requirements:
1. **Uniformity**: slots are evenly distributed
2. **Determinism**: same user always gets the same slot
3. **Independence**: group assignments across experiments are uncorrelated

In [ ]:
# Validate hash properties with 100,000 synthetic users
n_test_users = 100000
test_user_ids = [f"user_{i:08d}" for i in range(n_test_users)]

splitter_test = ABSplitter(count_slots=100)
splitter_test.split_experiments([
    ('exp_A', 50),
    ('exp_B', 50),
])

# 1. Check slot uniformity
slot_counts = np.zeros(100)
for uid in test_user_ids:
    slot = splitter_test._hash_to_slot(uid)
    slot_counts[slot] += 1

expected_per_slot = n_test_users / 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(100), slot_counts, color='steelblue', alpha=0.7)
axes[0].axhline(expected_per_slot, color='red', linestyle='--', linewidth=2, 
                label=f'Expected: {expected_per_slot:.0f}')
axes[0].set_xlabel('Slot Number')
axes[0].set_ylabel('User Count')
axes[0].set_title('Slot Distribution (100K users)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Check group balance within experiments
group_a_count = 0
group_b_count = 0
for uid in test_user_ids:
    assignment = splitter_test.process_user(uid)
    if assignment:
        exp_id = list(assignment.keys())[0]
        if assignment[exp_id] == 'A':
            group_a_count += 1
        else:
            group_b_count += 1

axes[1].bar(['Group A', 'Group B'], [group_a_count, group_b_count], 
            color=['#2196F3', '#FF9800'], alpha=0.8)
axes[1].axhline((group_a_count + group_b_count) / 2, color='red', linestyle='--', linewidth=2)
axes[1].set_ylabel('User Count')
axes[1].set_title('Group Balance Across All Experiments')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Statistical test for uniformity
chi2_stat, chi2_p = stats.chisquare(slot_counts)
print(f"Slot uniformity (chi-squared test):")
print(f"  Chi2 = {chi2_stat:.2f}, p = {chi2_p:.4f}")
print(f"  Max deviation from expected: {np.max(np.abs(slot_counts - expected_per_slot)):.0f} users")
print(f"\nGroup balance:")
print(f"  Group A: {group_a_count:,} ({group_a_count/(group_a_count+group_b_count)*100:.1f}%)")
print(f"  Group B: {group_b_count:,} ({group_b_count/(group_a_count+group_b_count)*100:.1f}%)")

In [ ]:
# 3. Verify determinism: same user always gets same assignment
print("Determinism check (same user, called 5 times):")
for uid in ['user_12345', 'user_99999', 'quickcart_vip_42']:
    assignments = [splitter_test.process_user(uid) for _ in range(5)]
    all_same = all(a == assignments[0] for a in assignments)
    print(f"  {uid}: {assignments[0]} -- consistent: {all_same}")

# 4. Verify independence: slot assignment doesn't predict group assignment
print(f"\nIndependence check across experiments:")

# Among users in exp_A slots (0-49), check group balance
# Among users in exp_B slots (50-99), check group balance
exp_a_groups = {'A': 0, 'B': 0}
exp_b_groups = {'A': 0, 'B': 0}

for uid in test_user_ids:
    assignment = splitter_test.process_user(uid)
    if 'exp_A' in assignment:
        exp_a_groups[assignment['exp_A']] += 1
    elif 'exp_B' in assignment:
        exp_b_groups[assignment['exp_B']] += 1

total_a = sum(exp_a_groups.values())
total_b = sum(exp_b_groups.values())
print(f"  exp_A: A={exp_a_groups['A']/total_a:.3f}, B={exp_a_groups['B']/total_a:.3f}")
print(f"  exp_B: A={exp_b_groups['A']/total_b:.3f}, B={exp_b_groups['B']/total_b:.3f}")
print(f"  Both close to 50/50 -- group assignment is independent of slot.")

:::{admonition} Why double-hashing? (click to expand)
:class: dropdown

The **double-hash** design separates two concerns:

**First hash** (`MD5(user_id) -> slot`): Determines *which experiment* a user is in. This depends only on the user's identity, not on which experiments exist. If we add or remove experiments (changing slot allocations), users don't move between existing experiments -- they only enter or leave newly allocated/deallocated slots.

**Second hash** (`MD5(user_id + experiment_id) -> group`): Determines the user's group *within* their experiment. Including the experiment_id means:
- If a user participates in sequential experiments, their group assignments are independent
- There's no systematic bias (e.g., "users who were in treatment for Experiment 1 are always in treatment for Experiment 2")

**Why MD5?** We don't need cryptographic security -- just uniform distribution. MD5 is fast and has excellent avalanche properties (small input changes produce completely different outputs). For production systems handling billions of assignments per day, this matters.

**Alternative: salt-based approach**
Some platforms use `hash(salt + user_id)` where the salt changes per experiment. This is equivalent to our approach but makes the "salt = experiment_id" relationship explicit.
:::

---

## 10. Handling Multiple Concurrent Experiments

With 10+ experiments running simultaneously, the platform team needs to:

1. **Prevent overlap**: Experiments testing the same surface should not share users
2. **Maximize utilization**: Don't leave traffic idle
3. **Enable cross-experiment analysis**: Detect interaction effects

### QuickCart's Experiment Management Scenario

The team wants to run:
- 2 search experiments (need isolation from each other)
- 1 checkout experiment (independent of search)
- 3 recommendation experiments (need isolation from each other)
- 1 pricing experiment (affects everything -- needs its own traffic)

In [ ]:
# QuickCart's full experiment allocation
splitter_prod = ABSplitter(count_slots=100)

# Allocate experiments across traffic
production_experiments = [
    ('search_ranking_v3', 15),        # 15% traffic
    ('search_ui_refresh', 15),        # 15% traffic  
    ('checkout_one_click', 20),       # 20% traffic
    ('rec_collaborative', 10),        # 10% traffic
    ('rec_content_based', 10),        # 10% traffic
    ('rec_hybrid', 10),              # 10% traffic
    ('dynamic_pricing', 10),          # 10% traffic
    # 10% unallocated (reserved for emergency experiments)
]

allocations = splitter_prod.split_experiments(production_experiments)
splitter_prod.get_allocation_summary()

# Visualize slot allocation
fig, ax = plt.subplots(figsize=(14, 3))

colors_map = {
    'search_ranking_v3': '#1f77b4',
    'search_ui_refresh': '#4ca3dd',
    'checkout_one_click': '#2ca02c',
    'rec_collaborative': '#ff7f0e',
    'rec_content_based': '#ffbb78',
    'rec_hybrid': '#d62728',
    'dynamic_pricing': '#9467bd',
}

for exp_id, (start, end) in allocations.items():
    width = end - start + 1
    ax.barh(0, width, left=start, height=0.5, 
            color=colors_map.get(exp_id, 'gray'), alpha=0.8,
            edgecolor='white', linewidth=1)
    ax.text(start + width/2, 0, exp_id.replace('_', '\n'), 
            ha='center', va='center', fontsize=8, fontweight='bold')

# Show unallocated
allocated_end = max(end for _, (_, end) in allocations.items())
if allocated_end < 99:
    ax.barh(0, 99 - allocated_end, left=allocated_end + 1, height=0.5,
            color='lightgray', alpha=0.5, edgecolor='white', linewidth=1)
    ax.text(allocated_end + 1 + (99 - allocated_end)/2, 0, 'Reserved',
            ha='center', va='center', fontsize=8, color='gray')

ax.set_xlim([-0.5, 100.5])
ax.set_xlabel('Traffic Slot')
ax.set_yticks([])
ax.set_title('QuickCart Experiment Traffic Allocation')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Simulate 50,000 users through the production experiment system
n_users_sim = 50000
user_assignments = []

for i in range(n_users_sim):
    uid = f"quickcart_user_{i:07d}"
    assignment = splitter_prod.process_user(uid)
    user_assignments.append({
        'user_id': uid,
        'experiment': list(assignment.keys())[0] if assignment else 'none',
        'group': list(assignment.values())[0] if assignment else 'none'
    })

df_assignments = pd.DataFrame(user_assignments)

# Summary statistics
print("Production Experiment Assignment Summary (50K users):")
print("=" * 65)
exp_counts = df_assignments.groupby('experiment').size().sort_values(ascending=False)

print(f"\n{'Experiment':<25} {'Users':<8} {'% Traffic':<10} {'Group A':<10} {'Group B':<10}")
print("-" * 65)

for exp in exp_counts.index:
    n = exp_counts[exp]
    pct = n / n_users_sim * 100
    if exp != 'none':
        exp_data = df_assignments[df_assignments['experiment'] == exp]
        n_a = (exp_data['group'] == 'A').sum()
        n_b = (exp_data['group'] == 'B').sum()
        print(f"{exp:<25} {n:<8} {pct:<10.1f} {n_a:<10} {n_b:<10}")
    else:
        print(f"{exp:<25} {n:<8} {pct:<10.1f} {'--':<10} {'--':<10}")

# Verify no user is in multiple experiments
users_in_experiments = df_assignments[df_assignments['experiment'] != 'none']
users_multi = users_in_experiments.groupby('user_id').size()
assert (users_multi <= 1).all(), "Some users are in multiple experiments!"
print(f"\nValidation: No user appears in more than one experiment (slot isolation works).")

In [ ]:
# Putting it all together: run an experiment through the full pipeline
# 1. Assign users via hash-based splitting
# 2. Simulate metrics (some with real effects)
# 3. Apply tiered multiple testing corrections

np.random.seed(2024)

# Focus on checkout_one_click experiment
exp_name = 'checkout_one_click'
exp_users_a = []
exp_users_b = []

for i in range(100000):
    uid = f"quickcart_user_{i:07d}"
    assignment = splitter_prod.process_user(uid)
    if exp_name in assignment:
        if assignment[exp_name] == 'A':
            exp_users_a.append(uid)
        else:
            exp_users_b.append(uid)

n_a, n_b = len(exp_users_a), len(exp_users_b)
print(f"Checkout One-Click Experiment:")
print(f"  Group A (control):   {n_a:,} users")
print(f"  Group B (treatment): {n_b:,} users")

# Simulate 12 metrics with tiered classification
metric_config = {
    # Primary metrics (use Holm)
    'revenue': {'type': 'primary', 'effect': 0.08},         # Real effect
    'conversion': {'type': 'primary', 'effect': 0.12},      # Real effect
    'aov': {'type': 'primary', 'effect': 0.0},              # No effect
    # Guardrail metrics (use Bonferroni)  
    'page_load_ms': {'type': 'guardrail', 'effect': 0.0},
    'error_rate': {'type': 'guardrail', 'effect': 0.0},
    'bounce_rate': {'type': 'guardrail', 'effect': 0.0},
    # Secondary metrics (use BH FDR)
    'pages_per_session': {'type': 'secondary', 'effect': 0.05},
    'time_on_site': {'type': 'secondary', 'effect': 0.0},
    'search_ctr': {'type': 'secondary', 'effect': 0.0},
    'add_to_cart': {'type': 'secondary', 'effect': 0.07},   # Real effect
    'wishlist_adds': {'type': 'secondary', 'effect': 0.0},
    'return_visits': {'type': 'secondary', 'effect': 0.0},
}

# Generate synthetic metric data and compute p-values
pvalues_by_type = {'primary': [], 'guardrail': [], 'secondary': []}
all_results = []

for metric_name, config in metric_config.items():
    control_data = np.random.normal(0, 1, n_a)
    treatment_data = np.random.normal(config['effect'], 1, n_b)
    _, p = stats.ttest_ind(treatment_data, control_data)
    effect_obs = treatment_data.mean() - control_data.mean()
    
    pvalues_by_type[config['type']].append(p)
    all_results.append({
        'metric': metric_name, 'type': config['type'],
        'true_effect': config['effect'], 'observed_effect': effect_obs,
        'p_value': p
    })

# Apply tiered corrections
primary_pvals = np.array(pvalues_by_type['primary'])
guardrail_pvals = np.array(pvalues_by_type['guardrail'])
secondary_pvals = np.array(pvalues_by_type['secondary'])

primary_holm = holm_correction(primary_pvals, alpha=0.05)
guardrail_bonf = bonferroni_correction(guardrail_pvals, alpha=0.05)
secondary_bh = benjamini_hochberg(secondary_pvals, q=0.10)

# Merge results
type_idx = {'primary': 0, 'guardrail': 0, 'secondary': 0}
corrections_map = {
    'primary': primary_holm,
    'guardrail': guardrail_bonf,
    'secondary': secondary_bh
}
method_names = {'primary': 'Holm', 'guardrail': 'Bonferroni', 'secondary': 'BH'}

print(f"\n{'='*85}")
print(f"TIERED ANALYSIS: Checkout One-Click Experiment")
print(f"{'='*85}")
print(f"\n{'Metric':<20} {'Type':<12} {'Effect':<9} {'p-value':<10} {'Method':<12} {'Significant':<12}")
print("-" * 75)

for res in all_results:
    mtype = res['type']
    idx = type_idx[mtype]
    correction = corrections_map[mtype]
    is_sig = correction['rejections'][idx]
    method = method_names[mtype]
    type_idx[mtype] += 1
    
    sig_mark = '  *' if is_sig else ''
    print(f"{res['metric']:<20} {mtype:<12} {res['observed_effect']:>+.4f}  "
          f"{res['p_value']:<10.4f} {method:<12} {sig_mark}")

---

## Summary

### Multiple Testing Corrections

| Method | Controls | Formula | Use When |
|--------|----------|---------|----------|
| **Bonferroni** | FWER | Reject if $p_i < \alpha/m$ | Guardrail metrics; any false positive is costly |
| **Holm's Step-Down** | FWER | Sequential thresholds $\alpha/(m-k+1)$ | Primary metrics; always prefer over Bonferroni |
| **Benjamini-Hochberg** | FDR | Compare sorted $p_{(k)}$ to $(k/m) \cdot q$ | Secondary/exploratory metrics |

### Experiment Infrastructure

| Component | Implementation | Purpose |
|-----------|---------------|----------|
| **Slot allocation** | 100 slots, contiguous ranges per experiment | Traffic isolation between experiments |
| **First hash** | `MD5(user_id) % 100` | Deterministic slot assignment |
| **Second hash** | `MD5(user_id + exp_id) % 2` | Independent group assignment |
| **Double-hashing** | Separate slot and group decisions | Independence across experiments |

### Key Principles

| Principle | Explanation |
|-----------|-------------|
| FWER vs FDR | Choose based on the cost of false positives vs false negatives |
| Holm dominates Bonferroni | Always use Holm unless simplicity is paramount |
| Hash-based splitting | Deterministic, scalable, no database lookup at assignment time |
| Slot isolation | Prevents cross-experiment contamination for same-surface tests |
| Metric categorization | Different correction methods for different metric roles |

### What's Next

In **Week 8**, we address the peeking problem -- what happens when you check experiment results repeatedly before the planned sample size. We'll implement Wald's Sequential Probability Ratio Test (SPRT) for principled continuous monitoring that controls both Type I and Type II error rates.